In [23]:
import pandas as pd
import os
import re

In [25]:
# 1. Dosyaların bulunduğu klasör
folder_path = r"C:\Users\Bedirhan\Desktop\ARSEK\saglik-personel-yer-secimi\veriler\model2023\birlestirme"  # Burayı değiştireceğiz

In [26]:
# 2. Tüm dosyaları listele
files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

In [29]:
# 3. Ana DataFrame
all_data = []

In [30]:
# 4. Dosyaları tek tek işle
for file in files:
    file_path = os.path.join(folder_path, file)
    
    # Dosya adından veri_tipi, kategori ve kurum bilgisi çıkar
    if file.startswith('hastane-personel'):
        veri_tipi = 'personel'
    elif file.startswith('hastane-sayisi'):
        veri_tipi = 'hastane_sayisi'
    elif file.startswith('hastane-yatak'):
        veri_tipi = 'yatak_sayisi'
    elif file.startswith('nufus'):
        veri_tipi = 'nufus'
    else:
        continue  # bilmediğimiz dosya varsa atla
    
    # Kategori ve kurum bilgisi çıkarma
    parts = file.replace('.csv','').split('-')
    
    if veri_tipi == 'personel':
        kategori = parts[2]  # asistan, ebe, hemşire vs.
        kurum = parts[3]     # devlet, ozel, universite
    elif veri_tipi == 'hastane_sayisi':
        kategori = 'hastane' # sabit kategori
        kurum = parts[2]
    elif veri_tipi == 'yatak_sayisi':
        kategori = 'yatak'
        kurum = parts[2]
    elif veri_tipi == 'nufus':
        kategori = 'nufus'
        kurum = 'yok'
    
    # CSV'yi oku
    df = pd.read_csv(file_path, encoding='utf-8')
    
    # Şehri ilk kolondan al, yılları sütunlardan al
    df = df.rename(columns={df.columns[0]: 'il_adı'})
    df_long = pd.melt(df, id_vars=['il_adı'], var_name='yıl', value_name='değer')

    # Yeni sütunlar ekle
    df_long['veri_tipi'] = veri_tipi
    df_long['kategori'] = kategori
    df_long['kurum'] = kurum
    
    # Listeye ekle
    all_data.append(df_long)

In [31]:
# 5. Hepsini birleştir
final_df = pd.concat(all_data, ignore_index=True)

In [33]:
# 6. Veri tipi düzeltmeleri
final_df['yıl'] = final_df['yıl'].astype(int)
final_df['değer'] = pd.to_numeric(final_df['değer'], errors='coerce')

In [34]:
# Kaydet: Python için
final_df.to_csv('all_data.csv', index=False, encoding='utf-8-sig')

In [37]:
print("Tüm dosyalar başarıyla ana_dataset.csv içinde birleşti!")

Tüm dosyalar başarıyla ana_dataset.csv içinde birleşti!
